In [1]:
import json
import asyncio
import httpx

In [2]:
# get_request
async def send_esri_request(url, params):
    async with httpx.AsyncClient() as client:
        response = await client.get(url, params=params)

    # check response
    if response.status_code == 200:
        print(f'response ok on {url}')
        
        json_response = response.json()
        print(json_response)
    else:
        print(f"Request failed with status code {response.status_code}")

    return response.json()

async def gather_task(url, list_id):
    tasks = [send_esri_request(f'{url}{i}?f==pjson') for i in list_id]
    result_all_json = await asyncio.gather(*tasks)
    return result_all_json

def converting_json_geojson(list_all_json):
    geojs = {   "type": "FeatureCollection",
               "features":  [
          {
               "type": "Feature",
               "geometry": {
                    "type": "Polygon",
                    "coordinates": d['feature']['geometry']['rings'],
               },
               "properties": {j:v for j,v in d['feature']['attributes'].items()},
          }
            for d in list_all_json
          ],
          }
    return geojs

In [3]:
# input
year  = 2012 
extent_boundary = '{xmin:107.593163470013,ymin:-7.13806689350134,xmax:107.685427020972,ymax:-7.06496818910933}' 
id_object_name = 'OBJECTID'

####

In [4]:
if year == 2022:
    base_url = "https://geoportal.menlhk.go.id/server/rest/services/SIGAP_Interaktif/Penutupan_Lahan_2022/MapServer/0/"

elif year == 2012:
    base_url = 'https://geoportal.menlhk.go.id/server/rest/services/Time_Series/PL_2012/MapServer/0/'
    

print(base_url)

params = {
    "where": f"{id_object_name} >= -1",
    "text": "",
    "objectIds": "",
    "time": "",
    "timeRelation": "esriTimeRelationOverlaps",
    "geometry": extent_boundary,
    "geometryType": "esriGeometryEnvelope",
    "inSR": 4326,
    "spatialRel": "esriSpatialRelIntersects",
    "units": "esriSRUnit_Foot",
    "outFields": "*",
    "returnGeometry": "false",
    "returnTrueCurves": "false",
    "maxAllowableOffset": "",
    "geometryPrecision": "",
    "outSR": 4326,
    "havingClause": "",
    "returnIdsOnly": "false",
    "returnCountOnly": "false",
    "orderByFields": "",
    "groupByFieldsForStatistics": "",
    "outStatistics": "",
    "returnZ": "false",
    "returnM": "false",
    "gdbVersion": "",
    "historicMoment": "",
    "returnDistinctValues": "false",
    "resultOffset": "",
    "resultRecordCount": "",
    "returnExtentOnly": "false",
    "sqlFormat": "none",
    "datumTransformation": "",
    "parameterValues": "",
    "rangeValues": "",
    "quantizationParameters": "",
    "featureEncoding": "esriDefault",
    "f": "pjson"
}

https://geoportal.menlhk.go.id/server/rest/services/Time_Series/PL_2012/MapServer/0/


In [5]:
# request get list id

result_list_id = await send_esri_request(f'{base_url}query', params)


ConnectTimeout: 

In [ ]:
list_id = [i['attributes'][id_object_name] for i in result_list_id['features']]
print(list_id)

In [158]:
all_json_list = await gather_task(base_url,list_id)
geojson = converting_json_geojson(all_json_list)

# geojson_2022
with open('./t4t_LC2012.geojson','w') as json_outfile:
    json.dump(geojson_2022,json_outfile)

In [ ]:
# request get list id
result_list_id_2022 = await send_esri_request(f'{base_url}query')
list_id_2022 = [i['attributes'][id_object_name] for i in result_list_id_2022['features']]
all_json_list_2022 = await gather_task(base_url,list_id_2022)
geojson_2022 = converting_json_geojson(all_json_list_2022)